# Sage benchmark search — NRP workspace

This standalone notebook optionally downloads all five benchmark datasets and returns up to **25 search results**. It either builds fresh vectors or restores an NPZ backup, then stores the vectors and caption text in an embedded SQLite database.

Retrieval fuses up to three legs. For **edge_v1 two are active**: a CLIP image vector, and **BM25 over the Gemma captions**. Captions are not embedded as dense vectors, because `baseline`, `v10`, `v11`, and `v12` all ran `clip_hybrid_query` against a single `clip` image vector with the caption text searched lexically — a dense caption leg would not be comparable to any of them.

The third leg is kept first-class for **edge_v2**, which will have caption vectors: build with `EMBED_CAPTIONS = True` and raise `CAPTION_WEIGHT`. Requesting a caption weight against an index with no caption vectors raises an error rather than silently reweighting, so two runs can never report the same weights while fusing differently.

The model ID is stored in the database so query vectors always use the matching encoder. Before the custom-query demo, the notebook generates fresh edge_v1 benchmark results and compares them with every bundled baseline/v10/v11/v12 per-query result. Existing edge_v1 and edge_v2 result files are never loaded.


## 1. Install this notebook's requirements

Run once after cloning the repository, then restart the kernel if Jupyter asks.

In [ ]:
from pathlib import Path

candidates = [
    Path.cwd(),
    Path.cwd() / "notebooks" / "ndp_workspace",
]
NOTEBOOK_DIR = next(path.resolve() for path in candidates if (path / "requirements.txt").is_file())
REQUIREMENTS = NOTEBOOK_DIR / "requirements.txt"
print(f"Notebook directory: {NOTEBOOK_DIR}")
%pip install -q -r {REQUIREMENTS}

## 2. Configuration

Copy `.env.example` to `.env` and put the Hugging Face token there. This cell then asks two explicit questions: whether benchmark data should be downloaded, and whether image vectors should be built or restored from an NPZ backup.

Default weights are `IMAGE_WEIGHT=0.40` / `BM25_WEIGHT=0.60`, matching the `alpha=0.4` used by the v10/v11/v12 reference runs (Weaviate's `alpha` is the vector share of a hybrid query).


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
load_dotenv(NOTEBOOK_DIR / ".env")

from notebook_helpers import (
    DATASETS,
    TextImageEncoder,
    benchmark_table,
    build_index,
    download_benchmarks,
    evaluate_benchmarks,
    extract_benchmark_images,
    load_and_resolve_captions,
    load_portable_index,
    load_reference_results,
    load_sqlite_vector_database,
    overall_table,
    query_result_table,
    save_sqlite_vector_database,
    search_index,
    show_results,
)

def ask_choice(prompt: str, choices: dict):
    while True:
        answer = input(prompt).strip().lower()
        if answer in choices:
            return choices[answer]
        print(f"Choose one of: {', '.join(choices)}")

HF_TOKEN = os.getenv("HF_TOKEN") or None
WORKSPACE_DATA = NOTEBOOK_DIR / "data"
HF_HOME = WORKSPACE_DATA / "huggingface"
os.environ["HF_HOME"] = str(HF_HOME)

DATASET_ROOT = WORKSPACE_DATA / "benchmarking" / "datasets"
IMAGE_ROOT = WORKSPACE_DATA / "benchmarking" / "images"
GEMMA_CAPTIONS_FILE = NOTEBOOK_DIR / "assets" / "gemma3_4b_it_edge_v1_benchmark_captions.jsonl"
RESOLVED_CAPTIONS_FILE = WORKSPACE_DATA / "gemma3_4b_it_edge_v1_benchmark_captions_resolved.jsonl"
# Backup mode defaults to the bundled edge_v1 index; build mode writes a
# separate model-named file so a build never overwrites that backup.
DEFAULT_BACKUP_FILE = WORKSPACE_DATA / "edge_v1_benchmarks.npz"
DATABASE_FILE = WORKSPACE_DATA / "vector_database" / "mobileclip2_benchmarks.sqlite3"
MOBILECLIP_MODEL_ID = "timm/MobileCLIP2-S0-OpenCLIP"
DEVICE = "auto"
BATCH_SIZE = 64
TOP_K = 25
# edge_v1 has two active legs: image vector + BM25 over captions. 0.40/0.60
# mirrors the alpha=0.4 hybrid split used by the v10/v11/v12 reference runs.
# CAPTION_WEIGHT stays 0.0 because edge_v1 has no caption vectors; an edge_v2
# index built with build_index(embed_captions=True) can raise it. Setting it
# above 0 against an index without caption vectors raises instead of silently
# reweighting.
IMAGE_WEIGHT = 0.40
CAPTION_WEIGHT = 0.0
BM25_WEIGHT = 0.60
EMBED_CAPTIONS = False  # True builds an edge_v2-style dense caption leg
GENERATED_BENCHMARK_ROOT = WORKSPACE_DATA / "generated_benchmarks"
REFERENCE_RESULT_ROOT = NOTEBOOK_DIR / "results" / "benchmarks"

DOWNLOAD_BENCHMARKS = ask_choice(
    "Download all benchmark datasets from Hugging Face? [yes/no]: ",
    {"yes": True, "y": True, "no": False, "n": False},
)
VECTOR_SOURCE = ask_choice(
    "Build image vectors or use an NPZ backup? [build/backup]: ",
    {"build": "build", "b": "build", "backup": "backup", "npz": "backup"},
)
if DOWNLOAD_BENCHMARKS and not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in ndp_workspace/.env before downloading.")
if VECTOR_SOURCE == "backup":
    entered = input(f"NPZ backup path [{DEFAULT_BACKUP_FILE}]: ").strip()
    BACKUP_FILE = Path(entered).expanduser() if entered else DEFAULT_BACKUP_FILE
    if not BACKUP_FILE.is_absolute():
        BACKUP_FILE = (NOTEBOOK_DIR / BACKUP_FILE).resolve()
else:
    BACKUP_FILE = (
        WORKSPACE_DATA / "backups"
        / f"{MOBILECLIP_MODEL_ID.rsplit('/', 1)[-1].lower()}_benchmarks.npz"
    )
    if not HF_TOKEN:
        raise RuntimeError("Set HF_TOKEN in ndp_workspace/.env before building image vectors.")
print({
    "download_benchmarks": DOWNLOAD_BENCHMARKS,
    "vector_source": VECTOR_SOURCE,
    "datasets": list(DATASETS),
    "npz_file": str(BACKUP_FILE),
    "sqlite_database": str(DATABASE_FILE),
    "top_k": TOP_K,
})

## 3. Download and materialize all benchmark images

The exact Hugging Face revisions are pinned in `notebook_helpers.py`. If the user answered **yes**, all five datasets are downloaded with `HF_TOKEN`. If the answer was **no**, the cell verifies that the standalone workspace already contains the Parquet files. Images are always materialized locally because benchmarking and visual search require them.

In [ ]:
if DOWNLOAD_BENCHMARKS:
    download_benchmarks(DATASET_ROOT, token=HF_TOKEN)
else:
    missing = [
        name for name in DATASETS
        if not list((DATASET_ROOT / name / "data").glob("*.parquet"))
    ]
    if missing:
        raise FileNotFoundError(
            f"Benchmark data is absent for {missing}. Rerun configuration and answer yes."
        )
image_counts = extract_benchmark_images(DATASET_ROOT, IMAGE_ROOT)
print(f"Total unique benchmark images: {sum(image_counts.values()):,}")

## 4. Build or restore vectors, then populate the local database

**Build** uses the bundled `gemma3_4b_it_edge_v1_benchmark_captions.jsonl` — the original Gemma 3 caption output — then embeds the corresponding **images** with `MOBILECLIP_MODEL_ID`. With `EMBED_CAPTIONS = False` (edge_v1) captions are stored as text only and searched by BM25; set it True for an edge_v2-style index that also embeds the captions with the same encoder. It does not launch Gemma or create substitute captions.

**Backup** restores vectors from the selected NPZ. The bundled `data/edge_v1_benchmarks.npz` is edge_v1's original `apple/DFN5B-CLIP-ViT-H-14-378` image index, so backup mode reproduces edge_v1 rather than running MobileCLIP2. Cell 5 reads the model ID back out of the database and loads the matching query encoder, so the two never mix. That file is gitignored — on a fresh clone, choose **build**.

Both choices populate the same embedded SQLite database — no container or database server — and the remainder of the notebook reloads its data from SQLite. The `caption_vector` column is NULL for an image-only index.


In [ ]:
if VECTOR_SOURCE == "build":
    records = load_and_resolve_captions(
        GEMMA_CAPTIONS_FILE, IMAGE_ROOT, RESOLVED_CAPTIONS_FILE
    )
    downloaded_image_count = sum(image_counts.values())
    print(
        f"Gemma 3 caption coverage: {len(records):,}/{downloaded_image_count:,} images "
        f"({len(records) / downloaded_image_count:.2%})"
    )
    portable_index = build_index(
        records=records,
        image_root=IMAGE_ROOT,
        output=BACKUP_FILE,
        model_id=MOBILECLIP_MODEL_ID,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        token=HF_TOKEN,
        embed_captions=EMBED_CAPTIONS,
    )
else:
    portable_index = load_portable_index(BACKUP_FILE)

save_sqlite_vector_database(portable_index, DATABASE_FILE)
index = load_sqlite_vector_database(DATABASE_FILE)

dataset_counts = {
    name: sum(record.dataset == name for record in index.records)
    for name in DATASETS
}
print(dataset_counts)

## 5. Load the matching query encoder

The model ID stored in the SQLite database selects the matching query encoder. This prevents a query model from being mixed with vectors created by another model.

In [ ]:
query_encoder = TextImageEncoder(index.model_id, device=DEVICE, token=HF_TOKEN)

## 6. Generate a fresh benchmark run

This creates a new `edge_v1` result from the local SQLite database and public relevance labels. It reports MRR, Success@25, Diversity@25, the two-metric primary score, and the three-metric score. Every baseline/v10/v11/v12 per-query result is bundled under `results/benchmarks`; saved edge_v1 and edge_v2 result directories are not reused.

In [ ]:
benchmark_weights = dict(
    image_weight=IMAGE_WEIGHT,
    caption_weight=CAPTION_WEIGHT,
    bm25_weight=BM25_WEIGHT,
)
RUN_LABEL = f"edge_v1 (generated; {index.model_id.rsplit('/', 1)[-1]})"

generated_summaries, generated_query_rows = evaluate_benchmarks(
    index=index,
    encoder=query_encoder,
    dataset_root=DATASET_ROOT,
    output_root=GENERATED_BENCHMARK_ROOT,
    top_k=TOP_K,
    batch_size=BATCH_SIZE,
    system_version=RUN_LABEL,
    **benchmark_weights,
)
reference_summaries, reference_query_rows = load_reference_results(REFERENCE_RESULT_ROOT)
comparison_rows = reference_summaries + generated_summaries
all_query_rows = reference_query_rows + generated_query_rows

import pandas as pd
ALL_RESULTS_FILE = GENERATED_BENCHMARK_ROOT / "all_comparison_query_results.csv"
pd.DataFrame(all_query_rows).to_csv(ALL_RESULTS_FILE, index=False)
print(f"Saved all {len(all_query_rows):,} per-query rows to {ALL_RESULTS_FILE}")

### FireBench — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "Firebench").round(4))
query_result_table(all_query_rows, "Firebench")

### CloudBench — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "Cloudbench").round(4))
query_result_table(all_query_rows, "Cloudbench")

### INQUIRE — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "INQUIRE").round(4))
query_result_table(all_query_rows, "INQUIRE")

### CommonObjectsBench — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "Commonobjectsbench").round(4))
query_result_table(all_query_rows, "Commonobjectsbench")

### SageBench — summary metrics and every per-query result

In [ ]:
display(benchmark_table(comparison_rows, "Sagebench").round(4))
query_result_table(all_query_rows, "Sagebench")

### Overall — equal weight across all five benchmarks

The overall table includes MRR, Success@25, Diversity@25, the two-metric primary score, and the equal-weight three-metric score. `benchmark_count` makes incomplete systems visible.

In [ ]:
overall_table(comparison_rows).round(4)

## 7. Custom query — 25 visual results

Enter any text query. Search uses the SQLite-backed image/caption records and fuses the image-vector leg, the optional caption-vector leg, and BM25 over the captions. Each result prints the total score with its two weighted components (which sum to it), the raw cosine and BM25 values, the caption, the image, the image ID, and the database path.

In [ ]:
QUERY = input("Enter your image-search query: ").strip()
if not QUERY:
    raise ValueError("Query cannot be empty")

results = search_index(
    index=index,
    encoder=query_encoder,
    query=QUERY,
    top_k=TOP_K,
    image_weight=IMAGE_WEIGHT,
    caption_weight=CAPTION_WEIGHT,
    bm25_weight=BM25_WEIGHT,
)
show_results(results, IMAGE_ROOT)
results[:3]